# Lab 1: Working with data — Data processing with Pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NgKhiem69/datascience_course/blob/main/labs/01-working-with-data/data-processing-with-pandas.ipynb)

**Mục tiêu:** đọc dữ liệu điểm thi Tin học đại cương (`final_grades.xlsx`), khám phá dữ liệu (EDA), lọc – thêm cột – nhóm – thống kê bằng Pandas và lưu kết quả ra file CSV.

## 0. Thiết lập môi trường (Google Colab)

Dữ liệu được đọc trực tiếp từ thư mục `data/` của repo GitHub qua đường dẫn *raw*, nên không cần upload file hay mount Google Drive.

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Đường dẫn raw tới thư mục data trên GitHub
GITHUB_USER, GITHUB_REPO, BRANCH = "NgKhiem69", "datascience_course", "main"
DATA_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/data"

def data_path(filename):
    """Trả về URL raw của một file trong thư mục data/ của repo."""
    return f"{DATA_URL}/{filename}"

# Kiểm tra đang chạy trên Colab hay máy local
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running on Colab:", IN_COLAB)
print("Data URL:", DATA_URL)

Running on Colab: False
Data URL: https://raw.githubusercontent.com/NgKhiem69/datascience_course/main/data


## 1. Đọc dữ liệu

In [2]:
import pandas as pd
import numpy as np

# Đọc file Excel trực tiếp từ GitHub vào DataFrame
grade_df = pd.read_excel(data_path("final_grades.xlsx"))
print("Kích thước dữ liệu:", grade_df.shape)
grade_df.head(10)

Kích thước dữ liệu: (1230, 2)


,Major,Grade
0,cdt,8.21
1,cdt,9.25
2,cdt,8.50
3,cdt,3.90
4,cdt,7.50
5,cdt,7.25
6,cdt,7.06
7,cdt,7.83
8,cdt,6.40
9,cdt,6.23


## 2. Khám phá dữ liệu (EDA)

### 2.1 Thông tin chung và thống kê mô tả

In [3]:
# Kiểu dữ liệu, số giá trị khác null của từng cột
grade_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Major   1230 non-null   str    
 1   Grade   1230 non-null   float64
dtypes: float64(1), str(1)
memory usage: 19.3 KB


In [4]:
# Thống kê mô tả cột số (Grade)
grade_df.describe()

,Grade
count,1230.000000
mean,6.238154
std,1.605444
min,2.290000
25%,5.040000
50%,6.230000
75%,7.400000
max,10.000000


In [5]:
# Thống kê mô tả cột phân loại (Major)
grade_df.describe(include="object")

,Major
count,1230
unique,20
top,cntt
freq,228


### 2.2 Kiểm tra giá trị thiếu và giá trị trùng lặp

In [6]:
missing_values = grade_df.isnull().sum()
print("Missing values in each column:\n", missing_values)

# Số dòng trùng lặp hoàn toàn (cùng ngành, cùng điểm) — với dữ liệu điểm đây là chuyện bình thường,
# nên ta chỉ đếm để tham khảo, KHÔNG xoá.
print("\nSố dòng trùng lặp:", grade_df.duplicated().sum())

Missing values in each column:
 Major    0
Grade    0
dtype: int64

Số dòng trùng lặp: 335


In [7]:
# Kiểm tra điểm có nằm trong thang 0–10 không
invalid = grade_df[(grade_df["Grade"] < 0) | (grade_df["Grade"] > 10)]
print("Số điểm nằm ngoài thang 0–10:", len(invalid))

Số điểm nằm ngoài thang 0–10: 0


### 2.3 Các ngành học

In [8]:
# Danh sách các ngành (giá trị duy nhất của cột Major)
major_names = grade_df["Major"].unique()
print(f"Có {len(major_names)} ngành:", major_names)

Có 20 ngành: <StringArray>
[ 'cdt', 'cnot', 'khhh', 'ckdl',  'ctm', 'cnhh', 'cnmt', 'ktck',  'vsh',
  'tdh', 'kttt', 'cnxd',  'ddt', 'cntt',  'kte', 'ckts', 'cnnl', 'khmt',
 'cbts', 'ttql']
Length: 20, dtype: str


In [9]:
# Số sinh viên của từng ngành, sắp xếp giảm dần
grade_df["Major"].value_counts()

Major
cntt    228
cnot    189
khhh    150
ktck    122
ddt     105
cnxd     71
tdh      68
cdt      65
khmt     60
ttql     50
cnhh     34
ctm      32
cnmt     22
ckdl     15
ckts     13
cnnl      2
vsh       1
kttt      1
kte       1
cbts      1
Name: count, dtype: int64

## 3. Lọc dữ liệu

Lọc các sinh viên thuộc nhóm ngành CNTT: `khmt`, `cntt`, `ttql`.

In [10]:
it_majors = ["khmt", "cntt", "ttql"]
it_students_df = grade_df[grade_df["Major"].isin(it_majors)]
print(f"#IT students/#Total: {len(it_students_df)}/{len(grade_df)}")
it_students_df.head()

#IT students/#Total: 338/1230


,Major,Grade
376,cntt,6.08
377,cntt,7.31
379,cntt,5.56
380,cntt,5.60
381,cntt,5.08


In [11]:
# Lọc theo nhiều điều kiện: sinh viên CNTT đạt điểm từ 9 trở lên
top_it = grade_df[grade_df["Major"].isin(it_majors) & (grade_df["Grade"] >= 9)]
print("Số SV nhóm CNTT đạt >= 9 điểm:", len(top_it))
top_it.sort_values("Grade", ascending=False).head(10)

Số SV nhóm CNTT đạt >= 9 điểm: 37


,Major,Grade
924,khmt,10.00
1184,ttql,10.00
972,khmt,9.81
980,khmt,9.81
480,cntt,9.75
522,cntt,9.58
387,cntt,9.56
979,khmt,9.56
558,cntt,9.56
540,cntt,9.50


## 4. Thêm cột mới

### 4.1 Cột `Result`: *Pass* nếu điểm ≥ 5, ngược lại *Fail*

In [12]:
grade_df["Result"] = grade_df["Grade"].apply(lambda x: "Pass" if x >= 5 else "Fail")
# Cách vector hoá (nhanh hơn apply với dữ liệu lớn):
# grade_df["Result"] = np.where(grade_df["Grade"] >= 5, "Pass", "Fail")
grade_df.head(10)

,Major,Grade,Result
0,cdt,8.21,Pass
1,cdt,9.25,Pass
2,cdt,8.50,Pass
3,cdt,3.90,Fail
4,cdt,7.50,Pass
5,cdt,7.25,Pass
6,cdt,7.06,Pass
7,cdt,7.83,Pass
8,cdt,6.40,Pass
9,cdt,6.23,Pass


### 4.2 Cột `MajorGroup`: nhóm ngành *IT* và *Non-IT*

In [13]:
grade_df["MajorGroup"] = grade_df["Major"].apply(lambda x: "IT" if x in it_majors else "Non-IT")

# Lọc tất cả các dòng thuộc nhóm IT
it_students_df = grade_df[grade_df["MajorGroup"] == "IT"]
it_students_df.head(10)

,Major,Grade,Result,MajorGroup
376,cntt,6.08,Pass,IT
377,cntt,7.31,Pass,IT
379,cntt,5.56,Pass,IT
380,cntt,5.60,Pass,IT
381,cntt,5.08,Pass,IT
382,cntt,5.08,Pass,IT
383,cntt,8.25,Pass,IT
384,cntt,8.65,Pass,IT
385,cntt,5.08,Pass,IT
386,cntt,7.71,Pass,IT


### 4.3 Cột `Letter`: quy đổi điểm chữ

Dùng `pd.cut` để chia điểm thành các khoảng (thang tín chỉ): F < 4 ≤ D < 5.5 ≤ C < 7 ≤ B < 8.5 ≤ A.

In [14]:
bins = [0, 4, 5.5, 7, 8.5, 10.01]
letters = ["F", "D", "C", "B", "A"]
grade_df["Letter"] = pd.cut(grade_df["Grade"], bins=bins, labels=letters, right=False)
grade_df.sample(10, random_state=42)

,Major,Grade,Result,MajorGroup,Letter
548,cntt,6.04,Pass,IT,C
704,ctm,3.40,Fail,Non-IT,F
244,cnot,7.06,Pass,Non-IT,B
552,cntt,7.88,Pass,IT,B
1164,tdh,5.15,Pass,Non-IT,D
1049,ktck,4.33,Fail,Non-IT,D
901,khhh,8.21,Pass,Non-IT,B
589,cntt,7.71,Pass,IT,B
101,khhh,6.83,Pass,Non-IT,C
128,cnhh,5.10,Pass,Non-IT,D


In [15]:
# Phân bố điểm chữ
grade_df["Letter"].value_counts().sort_index()

Letter
F    102
D    325
C    368
B    320
A    115
Name: count, dtype: int64

## 5. Nhóm và thống kê (`groupby`)

### 5.1 Thống kê theo nhóm ngành IT / Non-IT

In [16]:
group_stats = grade_df.groupby("MajorGroup")["Grade"].agg(
    count="count", mean="mean", median="median", std="std", min="min", max="max"
).round(2)
group_stats

,count,mean,median,std,min,max
MajorGroup,,,,,,
IT,338,6.84,7.00,1.64,2.54,10.0
Non-IT,892,6.01,5.94,1.53,2.29,10.0


In [17]:
# Tỉ lệ đạt (Pass rate) theo nhóm ngành
pass_rate_group = (grade_df["Result"] == "Pass").groupby(grade_df["MajorGroup"]).mean().mul(100).round(2)
pass_rate_group.rename("Pass rate (%)")

MajorGroup
IT        86.09
Non-IT    72.65
Name: Pass rate (%), dtype: float64

### 5.2 Thống kê theo từng ngành

In [18]:
major_stats = (
    grade_df.groupby("Major")
    .agg(
        students=("Grade", "count"),
        mean_grade=("Grade", "mean"),
        median_grade=("Grade", "median"),
        pass_rate=("Result", lambda s: (s == "Pass").mean() * 100),
    )
    .round(2)
    .sort_values("students", ascending=False)
)
major_stats

,students,mean_grade,median_grade,pass_rate
Major,,,,
cntt,228,6.74,6.94,85.53
cnot,189,6.40,6.38,84.13
khhh,150,6.63,6.81,84.00
ktck,122,5.95,6.06,71.31
ddt,105,5.36,5.15,58.10
cnxd,71,5.60,5.56,67.61
tdh,68,5.22,4.95,50.00
cdt,65,6.23,6.40,69.23
khmt,60,6.78,6.75,86.67


In [19]:
# Top 5 ngành có điểm trung bình cao nhất (chỉ xét ngành có >= 30 sinh viên để kết quả đáng tin cậy)
major_stats[major_stats["students"] >= 30].sort_values("mean_grade", ascending=False).head(5)

,students,mean_grade,median_grade,pass_rate
Major,,,,
ttql,50,7.37,8.25,88.00
khmt,60,6.78,6.75,86.67
cntt,228,6.74,6.94,85.53
khhh,150,6.63,6.81,84.00
cnot,189,6.40,6.38,84.13


### 5.3 Bảng chéo (pivot table / crosstab)

In [20]:
# Số sinh viên theo nhóm ngành x điểm chữ
pd.crosstab(grade_df["MajorGroup"], grade_df["Letter"], margins=True)

Letter,F,D,C,B,A,All
MajorGroup,,,,,,
IT,16,57,95,106,64,338
Non-IT,86,268,273,214,51,892
All,102,325,368,320,115,1230


In [21]:
# Tỉ lệ (%) điểm chữ trong mỗi nhóm ngành
pd.crosstab(grade_df["MajorGroup"], grade_df["Letter"], normalize="index").mul(100).round(1)

Letter,F,D,C,B,A
MajorGroup,,,,,
IT,4.7,16.9,28.1,31.4,18.9
Non-IT,9.6,30.0,30.6,24.0,5.7


## 6. Lưu dữ liệu đã xử lý

Trên Colab file được lưu vào thư mục làm việc `/content`. Nếu chạy trên Colab, file sẽ được tải xuống máy.

In [22]:
output_file = "final_grades_updated.csv"
grade_df.to_csv(output_file, index=False)
print("Đã lưu:", output_file)

if IN_COLAB:
    from google.colab import files
    files.download(output_file)

Đã lưu: final_grades_updated.csv


## 7. Nhận xét

- Dữ liệu gồm 1230 bài thi, 20 ngành, không có giá trị thiếu và tất cả điểm đều trong thang 0–10.
- Hãy đọc bảng ở mục 5 để so sánh điểm trung bình và tỉ lệ đạt giữa nhóm ngành CNTT và các ngành khác.
- Một số ngành chỉ có 1–2 sinh viên nên thống kê của các ngành này không có ý nghĩa đại diện.